In [0]:
# IMPORA AS BIBLIOTECAS
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    regexp_replace,
    when
)

# CONFIGURAÇÃO DOS CAMINHOS
volume_path = "/Volumes/mba/stage/dados_bruto/ANEEL"

In [0]:
# ============================================
# LEITURA DO ARQUIVO
# BANDEIRA TARIFÁRIA - ACIONAMENTO
# ============================================

df_bandeira_acionada = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/ANEEL-bandeira-tarifaria-acionamento.csv")
)

# ============================================
# TRANSFORMAÇÃO DOS TIPOS DE DADOS
# ============================================

df_bandeira_acionada = (
    df_bandeira_acionada
    .withColumn(
        "DatGeracaoConjuntoDados",
        to_date(col("DatGeracaoConjuntoDados"), "yyyy-MM-dd")
    )
    .withColumn(
        "DatCompetencia",
        to_date(col("DatCompetencia"), "yyyy-MM-dd")
    )
    .withColumn(
        "VlrAdicionalBandeira",
        regexp_replace(
            regexp_replace(col("VlrAdicionalBandeira"), "\\.", ""),
            ",",
            "."
        ).cast("decimal(20,2)")
    )
)

# ============================================
# GRAVAÇÃO NA DELTA TABLE
# ============================================

(
    df_bandeira_acionada.write
    .mode("overwrite")
    .saveAsTable("mba.raw.bandeira_acionada")
)

In [0]:
# ============================================
# LEITURA DO ARQUIVO
# BANDEIRA TARIFÁRIA - ADICIONAL
# ============================================

df_bandeira_adicional = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/ANEEL-bandeira-tarifaria-adicional.csv")
)

# ============================================
# TRANSFORMAÇÃO DOS TIPOS DE DADOS
# ============================================

df_bandeira_adicional = (
    df_bandeira_adicional
    .withColumn(
        "DatGeracaoConjuntoDados",
        to_date(col("DatGeracaoConjuntoDados"), "yyyy-MM-dd")
    )
    .withColumn(
        "DatVigencia",
        to_date(col("DatVigencia"), "yyyy-MM-dd")
    )
    .withColumn(
        "VlrAdicionalBandeiraRSMWh",
        regexp_replace(
            regexp_replace(
                col("VlrAdicionalBandeiraRSMWh"),
                "\\.",
                ""
            ),
            ",",
            "."
        ).cast("decimal(20,2)")
    )
)

# ============================================
# GRAVAÇÃO NA DELTA TABLE
# ============================================

(
    df_bandeira_adicional.write
    .mode("overwrite")
    .saveAsTable("mba.raw.bandeira_tarifaria_adicional")
)

In [0]:
# ============================================
# LEITURA DO ARQUIVO
# BANDEIRA TARIFÁRIA - Conta Bandeira
# ============================================

df_bandeira_adicional = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/ANEEL-bandeira-tarifaria-conta-bandeira.csv")
)

# ============================================
# FUNÇÃO PARA CONVERTER NÚMEROS
# ============================================

def converter_decimal(nome_coluna):
    return (
        when(
            trim(col(nome_coluna)) == "",
            None
        )
        .otherwise(
            regexp_replace(
                trim(col(nome_coluna)),
                ",",
                "."
            )
        )
        .cast("decimal(21,5)")
    )

# ============================================
# TRANSFORMAÇÃO DOS DADOS
# ============================================

df_conta_bandeira = (
    df_bandeira_adicional

    # ----------------------------------------
    # CAMPOS DE TEXTO E DATA
    # ----------------------------------------

    .select(

        # Datas
        to_date(
            trim(col("DatGeracaoConjuntoDados")),
            "yyyy-MM-dd"
        ).alias("DatGeracaoConjuntoDados"),

        # Sigla do agente
        trim(
            col("SigAgente")
        ).alias("SigAgente"),

        # ------------------------------------
        # MAPEAMENTO:
        # CSV = NumCNPJDistribuidora
        # DOCUMENTO = NumCPFCNPJ
        # ------------------------------------

        trim(
            col("NumCNPJDistribuidora")
        ).alias("NumCPFCNPJ"),

        # Data de competência
        to_date(
            trim(col("DatCompetencia")),
            "yyyy-MM-dd"
        ).alias("DatCompetencia"),

        # ------------------------------------
        # CAMPOS NUMÉRICOS
        # ------------------------------------

        converter_decimal(
            "VlrReceitaFaturada"
        ).alias("VlrReceitaFaturada"),

        converter_decimal(
            "VlrRepasseContaBandeira"
        ).alias("VlrRepasseContaBandeira"),

        converter_decimal(
            "VlrResultadoMCP"
        ).alias("VlrResultadoMCP"),

        converter_decimal(
            "VlrCCEARD"
        ).alias("VlrCCEARD"),

        # ------------------------------------
        # MAPEAMENTO DE NOME DO CSV PARA DOCUMENTO
        # ------------------------------------

        converter_decimal(
            "VlrRiscoHidrologicoCCGFRepactuadas"
        ).alias("VlrRiscoHidroCCGFRepactuadas"),

        converter_decimal(
            "VlrRiscoHidrologicoItaipu"
        ).alias("VlrRiscoHidrologicoItaipu"),

        converter_decimal(
            "VlrRiscoHidrologicoRepactuadas"
        ).alias("VlrRiscoHidrologicoRepactuadas"),

        converter_decimal(
            "VlrRiscoHidrologicoCCGF"
        ).alias("VlrRiscoHidrologicoCCGF"),

        converter_decimal(
            "VlrPrevisaoRiscoHidrologico"
        ).alias("VlrPrevisaoRiscoHidrologico"),

        # ------------------------------------
        # MAPEAMENTO:
        # CSV = VlrPremioDeRisco
        # DOCUMENTO = VlrPremiodeRisco
        # ------------------------------------

        converter_decimal(
            "VlrPremioDeRisco"
        ).alias("VlrPremiodeRisco"),

        converter_decimal(
            "VlrESSEER"
        ).alias("VlrESSEER"),

        converter_decimal(
            "VlrRessarcimentoCONER"
        ).alias("VlrRessarcimentoCONER"),

        converter_decimal(
            "VlrCVAEnergiaMesAnterior"
        ).alias("VlrCVAEnergiaMesAnterior"),

        converter_decimal(
            "VlrESSEERMesAnterior"
        ).alias("VlrESSEERMesAnterior"),

        converter_decimal(
            "VlrExposicaoInvoluntariaMesAnt"
        ).alias("VlrExposicaoInvoluntariaMesAnt"),

        converter_decimal(
            "VlrCVAEnergiaReceitaAlocada"
        ).alias("VlrCVAEnergiaReceitaAlocada"),

        converter_decimal(
            "VlrCVAESSEERReceitaAlocada"
        ).alias("VlrCVAESSEERReceitaAlocada"),

        converter_decimal(
            "VlrExposicaoInvolunReceitaAloc"
        ).alias("VlrExposicaoInvolunReceitaAloc"),

        converter_decimal(
            "VlrCVAEnergiaAposRepasse"
        ).alias("VlrCVAEnergiaAposRepasse"),

        converter_decimal(
            "VlrCVAESSEERAposRepasse"
        ).alias("VlrCVAESSEERAposRepasse"),

        converter_decimal(
            "VlrExposicaoInvolunAposRepasse"
        ).alias("VlrExposicaoInvolunAposRepasse"),

        converter_decimal(
            "VlrAjusteRepasse"
        ).alias("VlrAjusteRepasse")
    )
)

# ============================================
# CARGA DA DELTA TABLE
# ============================================

(
    df_conta_bandeira.write
    .mode("overwrite")
    .saveAsTable("mba.raw.conta_bandeira")
)

In [0]:
# Quantidade de registros

print(
    "bandeira_acionada:",
    spark.table("mba.raw.bandeira_acionada").count()
)

print(
    "bandeira_tarifaria_adicional:",
    spark.table("mba.raw.bandeira_tarifaria_adicional").count()
)

print(
    "conta_bandeira:",
    spark.table("mba.raw.conta_bandeira").count()
)